# Chunking + RAG Mechanism — Demo Notebook

Walks through the two pieces built for `US-001`/`US-002` (persistent memory) and the
general RAG mechanism requested afterward: **chunking**, **ingestion**, and
**retrieval** — using the real modules in `orchestration-service/app/`, not a
reimplementation.

| Module | Role |
|---|---|
| `app/chunking.py` | Splits raw text into overlapping windows |
| `app/rag_documents_repo.py` | SQL storage for documents + chunks (SQLite) |
| `app/rag_store.py` | Vector storage for chunk embeddings (Chroma) |
| `app/rag_manager.py` | Ties the above together: `ingest_text`, `retrieve`, `list_documents`, `delete_document` |

**Prerequisite:** the Agentic Service must be running on `localhost:8001` (used for
embedding calls) — start it with `uvicorn app.main:app --port 8001` from
`agentic-service/`.

This notebook writes to the same `Storage/` SQLite DB and Chroma store the running
services use, so anything ingested here is real, persistent data — the cleanup cell
near the end removes only the notebook's own demo document, leaving the real
knowledge base (see the last section) untouched.

In [1]:
import sys
from pathlib import Path

# Adjust this if your clone of the repo lives elsewhere.
ORCHESTRATION_SERVICE_DIR = Path('/Users/sallasujitha/WorkSpace/selfie-me/orchestration-service')
sys.path.insert(0, str(ORCHESTRATION_SERVICE_DIR))

from app import chunking, rag_documents_repo, rag_manager
import pandas as pd

pd.set_option('display.max_colwidth', 100)
print('Modules loaded OK.')

Modules loaded OK.


## 1. Chunking mechanism

`chunk_text` splits text into fixed-size, overlapping character windows — a placeholder default (see the docstring in `chunking.py`) until real document structure informs something smarter.

In [2]:
sample_text = (
    "Reflective listening means restating what the person said in your own words "
    "before offering any suggestion. Validation always comes before advice. "
    "A useful technique is naming the emotion and the underlying need, such as "
    "anger masking a need for respect. Motivational approaches ask open questions "
    "rather than lecturing, drawing out the person's own reasons for change. "
    "Small, concrete next steps beat large, vague ones — identity-based habits work "
    "because they tie the action to who the person wants to become, not just to a goal."
)

chunks_default = chunking.chunk_text(sample_text)
chunks_small = chunking.chunk_text(sample_text, chunk_size=120, overlap=30)

print(f'Input length: {len(sample_text)} chars')
print(f'Default (chunk_size=800, overlap=150): {len(chunks_default)} chunk(s)')
print(f'Smaller (chunk_size=120, overlap=30):  {len(chunks_small)} chunk(s)\n')

for i, c in enumerate(chunks_small):
    print(f'--- chunk {i} ({len(c)} chars) ---')
    print(c)
    print()

Input length: 531 chars
Default (chunk_size=800, overlap=150): 1 chunk(s)
Smaller (chunk_size=120, overlap=30):  6 chunk(s)

--- chunk 0 (120 chars) ---
Reflective listening means restating what the person said in your own words before offering any suggestion. Validation a

--- chunk 1 (120 chars) ---
g any suggestion. Validation always comes before advice. A useful technique is naming the emotion and the underlying nee

--- chunk 2 (120 chars) ---
emotion and the underlying need, such as anger masking a need for respect. Motivational approaches ask open questions ra

--- chunk 3 (120 chars) ---
proaches ask open questions rather than lecturing, drawing out the person's own reasons for change. Small, concrete next

--- chunk 4 (119 chars) ---
r change. Small, concrete next steps beat large, vague ones — identity-based habits work because they tie the action to

--- chunk 5 (81 chars) ---
ecause they tie the action to who the person wants to become, not just to a goal.



Notice consecutive chunks share a trailing/leading overlap region — that's what keeps a sentence spanning a chunk boundary from losing context on either side.

## 2. Ingesting a document (RAG intake)

`ingest_text` chunks, stores (SQLite), and embeds+indexes (Chroma) a piece of text under a `profile_email` scope — the same scope key used for per-user data elsewhere in this codebase. Identical text ingested twice is a no-op (content-hash dedup).

In [3]:
DEMO_SCOPE = 'notebook_demo'

result = rag_manager.ingest_text(
    profile_email=DEMO_SCOPE,
    text=sample_text,
    title='NVC + Motivational Interviewing snippet',
    source_type='text',
    chunk_size=150,
    overlap=30,
)
print('First ingest:', result['already_ingested'], '-', result['chunk_count'], 'chunks')

result_again = rag_manager.ingest_text(
    profile_email=DEMO_SCOPE,
    text=sample_text,
    title='NVC + Motivational Interviewing snippet',
    source_type='text',
    chunk_size=150,
    overlap=30,
)
print('Second ingest (same text):', result_again['already_ingested'], '- dedup worked as expected' if result_again['already_ingested'] else '- NOT deduped (unexpected)')

document_id = result['document']['document_id']
document_id

First ingest: False - 5 chunks
Second ingest (same text): True - dedup worked as expected


'doc_e4edea2beb86'

## 3. Retrieval mechanism (semantic + recency ranking)

`retrieve` embeds the query, pulls nearest chunks from Chroma (cosine distance), drops any chunk whose document was deleted after indexing, then re-ranks by a blend of semantic similarity and recency (`rag_manager.SEMANTIC_WEIGHT` / `RECENCY_WEIGHT`).

In [4]:
queries = [
    'how do I respond when someone is angry',
    'turning a goal into a daily habit',
]

for q in queries:
    print(f'Query: {q!r}')
    hits = rag_manager.retrieve(DEMO_SCOPE, q, top_k=2)
    df = pd.DataFrame(hits)[['score', 'chunk_index', 'content']]
    display(df)
    print()

Query: 'how do I respond when someone is angry'


,score,chunk_index,content
0,0.8525,1,"lways comes before advice. A useful technique is naming the emotion and the underlying need, suc..."
1,0.8156,2,"d for respect. Motivational approaches ask open questions rather than lecturing, drawing out the..."



Query: 'turning a goal into a daily habit'


,score,chunk_index,content
0,0.8610,4,"who the person wants to become, not just to a goal."
1,0.8503,3,"r change. Small, concrete next steps beat large, vague ones — identity-based habits work because..."


## 4. Cleanup

Removes the notebook's own demo document (and its chunk embeddings) so re-running this notebook doesn't accumulate duplicate demo data. This does **not** touch the real knowledge base in the next section.

In [5]:
rag_manager.delete_document(document_id, DEMO_SCOPE)

after_delete = rag_manager.retrieve(DEMO_SCOPE, 'how do I respond when someone is angry', top_k=5)
print('Chunks retrievable after delete:', len(after_delete), '(expected: 0)')

Chunks retrievable after delete: 0 (expected: 0)


## 5. The real knowledge base — 10 psychology books (`US-003`)

These were ingested via `orchestration-service/scripts/ingest_pdfs.py` from
`Docs/ASSETS/BOOKS/`, under the shared `rag_manager.KNOWLEDGE_BASE_SCOPE` — a
reserved scope key (not a real user profile) for knowledge that isn't owned by
any one person, per the comment in `rag_manager.py`.

In [6]:
docs = rag_manager.list_documents(rag_manager.KNOWLEDGE_BASE_SCOPE)
rows = []
for d in docs:
    chunk_count = len(rag_documents_repo.list_chunks(d['document_id'], rag_manager.KNOWLEDGE_BASE_SCOPE))
    rows.append({'title': d['title'], 'chunks': chunk_count, 'ingested_at': d['created_at']})

kb_df = pd.DataFrame(rows).sort_values('title').reset_index(drop=True)
print(f"Total documents: {len(kb_df)}, total chunks: {kb_df['chunks'].sum()}")
kb_df

Total documents: 10, total chunks: 4857


,title,chunks,ingested_at
0,Atomic-Habits-James-Clear,369,2026-09-23T06:25:10.472132+00:00
1,"Daniel Kahneman-Thinking, Fast and Slow",912,2026-09-23T06:25:27.702162+00:00
2,Feeling Good PDF,98,2026-09-23T06:26:03.833758+00:00
3,Man_s Search for Meaning_ Young - Viktor E. Frankl,192,2026-09-23T06:24:44.093697+00:00
4,Mihaly-Csikszentmihalyi-Flow,633,2026-09-23T06:26:08.618374+00:00
5,Mindset - Carol Dweck,416,2026-09-23T06:26:51.318200+00:00
6,Nonviolent Communication_ A Language of Life_ Life-Changing Tools for Healthy Relationships ( PD...,312,2026-09-23T06:27:11.631985+00:00
7,Self-Compassion PDF,140,2026-09-23T06:27:29.150217+00:00
8,emotional-intelligence-daniel-goleman,659,2026-09-23T06:27:36.752758+00:00
9,motivational-interviewing-third-edition-helping-people-change_compress,1126,2026-09-23T06:28:10.197635+00:00


## 6. Querying the knowledge base by the five response aspects (`US-003`)

`US-003` defines five ordered response aspects for the emotional support agent:
**Recognize → Validate → Reframe → Motivate → Anchor to meaning**. One representative
query per aspect, run against the real knowledge base, shows which book each aspect
actually draws from in practice.

In [7]:
aspect_queries = {
    'Recognize': 'what is the person really feeling underneath their words',
    'Validate': 'reflecting feelings back before giving advice',
    'Reframe': 'catastrophizing and all-or-nothing thinking',
    'Motivate': 'asking questions instead of lecturing to draw out someone\'s own reasons for change',
    'Anchor to meaning': 'finding meaning and purpose during suffering',
}

for aspect, q in aspect_queries.items():
    hits = rag_manager.retrieve(rag_manager.KNOWLEDGE_BASE_SCOPE, q, top_k=1)
    if not hits:
        print(f'{aspect}: no match')
        continue
    top = hits[0]
    print(f'=== {aspect} ===')
    print(f"query: {q!r}")
    print(f"-> {top['document_title']}  (score={top['score']})")
    print(top['content'][:300].strip().replace('\n', ' ') + '...')
    print()

=== Recognize ===
query: 'what is the person really feeling underneath their words'
-> Nonviolent Communication_ A Language of Life_ Life-Changing Tools for Healthy Relationships ( PDFDrive )  (score=0.8908)
ing. An expression of feeling might    be: “I feel disgusted.” 7. If you circled this number, we’re not in agreement. I don’t    consider “like hitting you” to be a feeling. To me, it    expresses what the speaker imagines doing, rather than    how the speaker is feeling. An expression of feeling mi...



=== Validate ===
query: 'reflecting feelings back before giving advice'
-> motivational-interviewing-third-edition-helping-people-change_compress  (score=0.8881)
n counseling, but his student Charles Truax (1966) coded audiotapes of Rogers’s sessions and found that he was differentially “reinforcing” certain kinds of client statements while letting others pass without reflection or comment. It is truly difficult to respond unconditionally to whatever clients...



=== Reframe ===
query: 'catastrophizing and all-or-nothing thinking'
-> Atomic-Habits-James-Clear  (score=0.8716)
ithout thinking, the more your brain is free to focus on other areas. Knowledge compounds. Learning one new idea won’t make you a genius, but a commitment to lifelong learning can be transformative. Furthermore, each book you read not only teaches you something new but also opens up different ways o...



=== Motivate ===
query: "asking questions instead of lecturing to draw out someone's own reasons for change"
-> motivational-interviewing-third-edition-helping-people-change_compress  (score=0.9128)
you to __________?”   “How serious or urgent does this feel to you?”   “What do you think has to change?”   “Complete this sentence: ‘I really must __________.’ ”    As we discuss in Chapter 14 there is more to this than simply asking such questions. There are also specific processes for reflecting...



=== Anchor to meaning ===
query: 'finding meaning and purpose during suffering'
-> Man_s Search for Meaning_ Young - Viktor E. Frankl  (score=0.9235)
at all, then there must be a meaning in suffering. Suffering is an ineradicable part of life, even as fate and death. Without suffering and death human life cannot be complete.    The way in which a man accepts his fate and all the suffering it entails, the way in which he takes up his cross, gives...



### A note on retrieval quality

Run this and you'll likely see 4 of the 5 aspects land on the expected book, and one
(commonly *Reframe*) land on a plausible-but-not-ideal match. That's the real, honest
behavior of today's mechanism — simple embedding similarity over fixed-size chunks,
with no query rewriting, no reranking, and no per-aspect metadata filtering yet. It's
not a bug to fix silently; it's exactly the gap `rag_manager.py`'s comments already
flag ("a real hybrid ranker lands with the RAG work later") and worth keeping visible
here rather than cherry-picking a query that always looks perfect.